In [1]:
import pickle
import numpy as np
import networkx as nx
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti, DecomposePauliZEvolution
from itertools import combinations

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate, CXGate, SwapGate
from qiskit.transpiler import PassManager
from qiskit.converters import dag_to_circuit, circuit_to_dag
from qiskit.circuit import Parameter

from qiskit.transpiler.passes import (
    HighLevelSynthesis, 
    InverseCancellation
)

from qopt_best_practices.transpilation.qaoa_construction_pass import QAOAConstructionPass
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
from qiskit_aer.primitives import SamplerV2 as Sampler

from qiskit_qaoa.utils.qaoa_circuit_utils import get_mixer_operator, state_prep
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples

In [2]:
with open('/lustre/scratch127/qpg/jc59/hubo/simulation.grid.compilation.trivial.extra0.constraint0.5.four0.0.six1.0.pkl', 'rb') as f:
    data = pickle.load(f)
    
N: int = 3
T: int = 3
n = int(np.ceil(np.log2(2*N+1)))
extra = 0

In [3]:
data.keys()

dict_keys(['full_hamiltonian', 'compiled_hamiltonian', 1, 2, 3, 4, 5, 6, 7, 8])

In [4]:
swap_depth = 6
edge_map = data[swap_depth]
edge_map

{0: 8, 1: 7, 2: 6, 3: 5, 4: 4, 5: 3, 6: 2, 7: 1, 8: 0}

In [5]:
old_hamiltonian = data['compiled_hamiltonian']
num_qubits = old_hamiltonian.num_qubits

extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)

In [6]:
np.any(old_hamiltonian.paulis.z, axis=0)

array([ True,  True,  True,  True,  True,  True,  True,  True, False])

In [7]:
properties = {}
def get_permutation(pass_, dag, time, property_set, count):
    properties["virtual_permutation_layout"] = property_set["virtual_permutation_layout"]

In [8]:
def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')

In [9]:
num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map

basis_gates=["sx", "x", "rz", "rzz", "cz", "id"]


print(f'Physical qubits: {num_physical_qubits}')

coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)
dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)


pm = PassManager(
    [
        HighLevelSynthesis(basis_gates=["PauliEvolution"]), # Not needed if set up circuit as PauliEvolutionGate
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            extended_swap_strat,
            edge_colouring,
            max_layers=swap_depth,
            perform_extra_swaps=bool(extra)
        ),
        SwapToFinalMapping(),
        DecomposePauliZEvolution(extended_swap_strat._coupling_map),
        HighLevelSynthesis(
            basis_gates=["sx", "x", "rz", "rzz", "cx", "id", "swap"], 
        ),
        InverseCancellation(gates_to_cancel=[CXGate(), SwapGate()]),
    ]
)

cost_qc = QuantumCircuit(num_physical_qubits)
cost_qc.append(PauliEvolutionGate(old_hamiltonian, time=Parameter("c")), [edge_map[i] for i in range(len(edge_map))])
tcost_qc = pm.run(cost_qc, callback=get_permutation)

Physical qubits: 9
09:44:21 - qiskit_qaoa.utils.transpiler_passes - INFO - Max layers needed to apply swap decompose: 6
09:44:21 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 0
09:44:21 - qiskit_qaoa.utils.transpiler_passes - INFO - []
09:44:21 - qiskit_qaoa.utils.transpiler_passes - INFO - Not implementing those gates


09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (8, 7)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (7, 6)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (8, 7)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (7, 6)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (7, 6)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (8, 7)
09:44:21 - qiskit_qaoa.utils.transpiler_passes - WARNING - No single neighbour qubit found. Apply a random cx to break loops. Chosen: (7, 6)
09:44:21 - qi

In [ ]:
backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates=basis_gates,
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map, **backend_options)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')
sampler = Sampler(seed=1).from_backend(backend)

In [ ]:
if not 2*N+1 == 2**(int(np.log2(2*N+1))):
    sp = state_prep(N,T)
    mixer = get_mixer_operator(N,T)
    print('Using Grover mixer and state prep')
else:
    sp = None
    mixer = None
    print('Using X mixer and Hadamard state prep')

In [ ]:
p = 1
construction_pass = QAOAConstructionPass(p, init_state=sp, mixer_layer=mixer)
construction_pass.property_set = properties
qaoa_circ = dag_to_circuit(construction_pass.run(circuit_to_dag(tcost_qc)))

# Now transpile to basis gates
basis_gates = ["x", "rz", "ry", "rzz", "cz", "id", "h", "cx"]
t_qaoa_circ = transpile(qaoa_circ, basis_gates=basis_gates)



In [ ]:
print_circuit_info(tcost_qc, 'Cost layer')
print(f'Cost layer has {tcost_qc.num_qubits} qubits')

print_circuit_info(t_qaoa_circ, 'QAOA circuit')
print(f'QAOA circuit has {t_qaoa_circ.num_qubits} qubits')

In [ ]:
tcost_qc.draw(idle_wires=False, fold=-1)

In [ ]:
sp = state_prep(N, T)
sp.decompose('circuit*').draw(fold=-1)

In [ ]:
t_qaoa_circ.draw(idle_wires=False, fold=-1)

In [ ]:
x = [0.0, 0.0]
assigned_circuit = t_qaoa_circ.assign_parameters(x, inplace=False)
sampler_results = sampler.run([assigned_circuit], shots=10000).result()

In [ ]:
from collections import Counter


In [ ]:
counter = Counter(sampler_results[0].data.c.get_counts())

In [ ]:
counter

In [ ]:
len(counter.keys())

In [ ]:
keys = list(counter.keys())
keys.index('000010001')

In [ ]:
(2*N+1)**T

In [ ]:
t_sp = transpile(sp, basis_gates=basis_gates)
t_sp.measure_all()
sp_results = sampler.run([t_sp], shots=100000).result()
sp_counter = Counter(sp_results[0].data.meas.get_counts())

In [ ]:
len(sp_counter.keys())

In [ ]:
hamiltonian = old_hamiltonian.apply_layout([edge_map[i] for i in range(num_qubits)], num_physical_qubits)

In [ ]:
keys = list(counter.keys())
scores = evaluate_sparse_pauli_samples(keys, hamiltonian)
min_idx = (np.argmin(scores))
min_idxs = np.nonzero(scores == np.min(scores))[0]
for min_idx in min_idxs:
    key = keys[min_idx]
    unmapped = ''.join([key[edge_map[x]] for x in range(len(key))])
    unmapped_slices = [unmapped[i*n:(i+1)*n] for i in range(T)]
    path = [(sum(int(slice[i]) * 2**i for i in range(n))//2, sum(int(slice[i]) * 2**i for i in range(n)) % 2) for slice in unmapped_slices]
    print(path)
# 001011101 

In [ ]:
best = '001011101'
unmapped = ''.join([best[edge_map[x]] for x in range(len(best))])
unmapped # 101110100 -> 5, 3, 1 -> u2-,u1-,u0-

In [ ]:
from functools import reduce
from qiskit_qaoa.utils.hamiltonian_utils import indices_to_pauli
from qiskit_qaoa.utils.string_utils import bin_rep

In [ ]:
weights = [1,1,1,1,1,1]
V = 6
obj_spo = reduce(
    SparsePauliOp._add,
    [
        (
            reduce(
                SparsePauliOp._add,
                [
                    reduce(
                        SparsePauliOp.compose,
                        [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(i, n)[k]) * indices_to_pauli(t, k, n, T)) for k in range(n)]
                    ) + reduce(
                        SparsePauliOp.compose,
                        [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(i+1, n)[k]) * indices_to_pauli(t, k, n, T)) for k in range(n)]
                    )
                    for t in range(T)
                ],
                SparsePauliOp('I'  * n * T, np.array([0]))
            ) 
            - SparsePauliOp('I' * n * T, [weights[i]]) 
        ) ** 2
        for i in range(0, V, 2)
    ],
    SparsePauliOp('I'  * n * T, np.array([0]))
)

In [ ]:
obj_spo = obj_spo.simplify()
obj_spo

In [ ]:
cons_spo = old_hamiltonian - obj_spo
cons_spo = cons_spo.simplify()

In [ ]:
keys = list(counter.keys())
scores = evaluate_sparse_pauli_samples(keys, obj_spo)
min_idx = np.argmin(scores)
min_idxs = np.nonzero(scores == np.min(scores))
# for min_idx in min_idxs[0][:10]:
#     print(min_idx, scores[min_idx], keys[min_idx])
# 110100011 -> 3, 1, 6 N
# 010001100 -> 2, 4, 1 Y
# 001010000 -> 4, 2, 0 Y
# 001100110 -> 
print(np.min(scores))
for min_idx in min_idxs[0][:10]:
    key = keys[min_idx]
    slices = [key[i*3:(i+1)*3] for i in range(3)]
    vals = [sum(int(slice[i]) * 2**i for i in range(3)) // 2 for slice in slices]
    parities = [sum(int(slice[i]) * 2**i for i in range(3)) % 2 for slice in slices]
    print(key)
    print(vals)
    print(parities)
    print()

In [ ]:
keys= ['000000000', '000000010', '000000001', '000000011', '110100011', '000010001']
evaluate_sparse_pauli_samples(keys, obj_spo)

In [ ]:
keys = list(counter.keys())
scores = evaluate_sparse_pauli_samples(keys, cons_spo)
min_idx = (np.argmin(scores))
print(min_idx, scores[min_idx], keys[min_idx])
# 001011010100 -> 4, 6, 2, 1 -> u3+, end, u1+, u0- 

In [ ]:
scores[0:10]

In [ ]:
for key in counter.keys():
    key_slices = [key[3*i:3*(i+1)] for i in range(4)]
    if any(slice == '111' for slice in key_slices):
        print(key)

In [ ]:
sol = '000000010000000000000010000100000000000000000001000000100000010000000000000010000000000000000100001000000000000000001000000001000000100000000000'
sol = [int(c) for c in sol]
[idx for idx in range(len(sol)) if sol[idx] > 0]

In [ ]:
sol_slices = [sol[12*i:12*(i+1)] for i in range(12)]
sol_slices

In [ ]:
from pysat.formula import IDPool
num_nodes_g2: int = 12
variable_pool = IDPool(start_from=0)
variables = np.array(
    [
        [variable_pool.id(f"v_{i}_{j}") for j in range(num_nodes_g2)]
        for i in range(12)
    ],
    dtype=int,
)
vid2mapping = {v: idx for idx, v in np.ndenumerate(variables)}

In [ ]:
[vid2mapping[idx] for idx in range(len(sol)) if sol[idx] > 0]

In [ ]:
ess = ExtendedSwapStrategy.from_grid(3,4)

In [ ]:
3//2

In [ ]:
ess.distance_matrix

In [ ]:
import networkx as nx
g = nx.grid_2d_graph(3,3)
pos = nx.spring_layout(g)
nx.draw(g, pos=pos)
nodes = list(g.nodes)
labels = {nodes[x]: x for x in range(len(nodes))}
nx.draw_networkx_labels(g, pos, {n:lab for n,lab in labels.items() if n in pos})

In [ ]:
ess._swap_layers

In [ ]:
ess.distance_matrix